# Microsoft Agent Framework Demo
Please visit https://aka.ms/agentframework for the official code samples

In [38]:
from agent_framework import ChatAgent, AgentProtocol, AgentThread, HostedMCPTool
from agent_framework.azure import AzureAIAgentClient
from azure.identity.aio import AzureCliCredential
from typing import Any
from IPython.display import display, Markdown

### Basic example

In [39]:
async def main():
    async with (
        AzureCliCredential() as credential,
        ChatAgent(
            chat_client=AzureAIAgentClient(async_credential=credential),
            instructions="You are good at telling jokes."
        ) as agent,
    ):
        result = await agent.run("Tell me a joke about a pirate.")
        display(Markdown(result.text))

await main()

Why did the pirate go to the seafood restaurant?  
Because he heard they had great fish ‘n ships! Arrr! 🏴‍☠️

### Azure AI Agent with Remote MCP Example

In [ ]:
async def handle_approvals_with_thread(query: str, agent: "AgentProtocol", thread: "AgentThread"):
    """Here we let the thread deal with the previous responses, and we just rerun with the approval."""
    from agent_framework import ChatMessage

    result = await agent.run(query, thread=thread, store=True)
    while len(result.user_input_requests) > 0:
        new_input: list[Any] = []
        for user_input_needed in result.user_input_requests:
            print(
                f"User Input Request for function from {agent.name}: {user_input_needed.function_call.name}"
                f" with arguments: {user_input_needed.function_call.arguments}"
            )
            new_input.append(
                ChatMessage(
                    role="user",
                    contents=[user_input_needed.create_response(True)],
                ),
            )
        result = await agent.run(new_input, thread=thread, store=True)
    return result


async def main() -> None:
    """Example showing Hosted MCP tools for a Azure AI Agent."""
    async with (
        AzureCliCredential() as credential,
        AzureAIAgentClient(async_credential=credential) as chat_client,
    ):
        # commenting out azure-ai observability to supress warnings
        # await chat_client.setup_azure_ai_observability()
        agent = chat_client.create_agent(
            name="DocsAgent",
            instructions="You are a helpful assistant that can help with microsoft documentation questions.",
            tools=HostedMCPTool(
                name="Microsoft Learn MCP",
                url="https://learn.microsoft.com/api/mcp",
            ),
        )
        thread = agent.get_new_thread()
        # First query
        query1 = "How do I create an Azure storage account using az cli?"
        display(Markdown(f"**User:** {query1}"))
        result1 = await handle_approvals_with_thread(query1, agent, thread)
        display(Markdown(f"**{agent.name}:** {result1}"))
        display(Markdown("\n---\n"))
        # Second query
        query2 = "What is Microsoft Agent Framework?"
        display(Markdown(f"**User:** {query2}"))
        result2 = await handle_approvals_with_thread(query2, agent, thread)
        display(Markdown(f"**{agent.name}:** {result2}"))

await main()

**User:** How do I create an Azure storage account using az cli?

User Input Request for function from DocsAgent: microsoft_docs_search with arguments: {"query":"create Azure storage account using az cli"}


**DocsAgent:** To create an Azure storage account using the Azure CLI, use the following steps:

### 1. Sign in and prepare
- Make sure you are signed in to Azure CLI:
  ```bash
  az login
  ```
- Ensure you have an existing resource group or create one:
  ```bash
  az group create --name <your-resource-group> --location <region>
  ```

### 2. Create the storage account
Here is a typical command to create a standard general-purpose v2 storage account:

```bash
az storage account create \
  --name <unique-storage-account-name> \
  --resource-group <your-resource-group> \
  --location <region> \
  --sku Standard_RAGRS \
  --kind StorageV2 \
  --min-tls-version TLS1_2 \
  --allow-blob-public-access false
```

- `--name`: The storage account name must be unique globally.
- `--resource-group`: The resource group name you created or chose.
- `--location`: The Azure region for the storage account (e.g., eastus).
- You can adjust `--sku` for the type of redundancy (Standard_RAGRS, Standard_LRS, etc.)
- `--kind` usually is `StorageV2` for general-purpose v2.
- Other parameters are optional but recommended for security.

### Example

```bash
az storage account create \
  --name mystorageacct123456 \
  --resource-group myResourceGroup \
  --location eastus \
  --sku Standard_LRS \
  --kind StorageV2 \
  --min-tls-version TLS1_2 \
  --allow-blob-public-access false
```

For more details and options, you can visit the [official Microsoft documentation: Create an Azure storage account with Azure CLI](https://learn.microsoft.com/en-us/azure/storage/common/storage-account-create#create-a-storage-account).

Let me know if you want an example with specific features (like Data Lake, firewall settings, etc.)!


---


**User:** What is Microsoft Agent Framework?

User Input Request for function from DocsAgent: microsoft_docs_search with arguments: {"query":"Microsoft Agent Framework"}


**DocsAgent:** The Microsoft Agent Framework is a development framework provided by Microsoft for creating and orchestrating AI agents across different environments and services. It is designed to simplify the building, running, and coordination of intelligent agents powered by LLMs (Large Language Models), with a focus on type safety, extensibility, and integration with modern AI services such as Azure OpenAI, Azure AI Foundry, and more.

### Core Features

- **Agent Abstraction**: All agents are based on a common `AIAgent` base class. This allows developers to build agents that can represent anything from simple chatbots to sophisticated multi-agent orchestrations.
- **Service Flexibility**: Agents can use different underlying inference services (e.g., Azure OpenAI, OpenAI API, Azure AI Foundry) by plugging in various chat client implementations.
- **Multi-Agent Orchestration**: Supports higher-level orchestration across multiple agents, enabling complex workflows and collaborative scenarios.
- **Integration**: Connects easily to external services and APIs, and supports both synchronous and asynchronous messaging patterns.
- **Custom Agents**: You can build fully custom agents to control behavior or integrate specialized logic and tool usage.
- **Security/Compliance**: Offers guidelines and responsibility warnings for data flow, especially when using third-party providers.

### Example Agent Types

- **Azure AI Foundry Agent**: Uses Azure AI Foundry Agents Service.
- **Azure OpenAI ChatCompletion Agent**: Integrates Azure OpenAI's chat services.
- **OpenAI ChatCompletion Agent**: Uses OpenAI's own ChatCompletion API.
- **Custom and Proxy Agents**: Build your own or connect to remote agents using protocols like A2A.

### Programming Support

- Available in multiple languages, such as C# (.NET) and Python.
- Easily integrates with modern cloud and enterprise environments.

### Quick Start Example (C#)
```csharp
using Microsoft.Agents.AI;

var agent = new ChatClientAgent(chatClient, instructions: "You are a helpful assistant");
```

> For more technical docs and examples, visit the [Microsoft Agent Framework documentation](https://learn.microsoft.com/en-us/agent-framework/user-guide/agents/agent-types/).

---

Let me know if you want more in-depth samples or details about a specific use case!